In [ ]:
!pip install -q pydantic langchain-community langchain-openai --q

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["OPENAI_BASE_URL"] = "https://openai.vocareum.com/v1"
os.environ["LANGCHAIN_API_KEY"] = ""
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "Guardrails Demo v1"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

In [ ]:
import os
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.agents.middleware.pii import PIIDetectionError
from langchain_core.tools import tool

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
import re

In [ ]:
@tool
def process_order(customer_email: str, amount: float) -> str:
    """Processes an order for a customer given their email and total amount."""
    return f"Order processed for {customer_email} with total amount ${amount:.2f}."

tools = [process_order]

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
BANNED_WORDS = ["confidential", "internal_only", "classified", "secret_project"]
banned_words_pattern = r"(?i)\b(" + "|".join(re.escape(word) for word in BANNED_WORDS) + r")\b"

In [ ]:
agent = create_agent(
    model="gpt-4o-mini",
    tools=tools,
    middleware=[
        # Strategy 1: Redact email addresses in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
            apply_to_output=True,
        ),
        # Strategy 2: Mask credit cards (e.g., ****-****-****-1234)
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
            apply_to_output=True,
        ),
        # 3. Custom PII: Name Redaction using custom regex pattern
        # Matches common name prefixes like "My name is John Doe" or "I am Jane Smith"
        PIIMiddleware(
            "name",
            detector=r"(?i)(?:my name is|i am|customer:)\s+([A-Z][a-z]+\s+[A-Z][a-z]+)",
            strategy="redact",
            apply_to_input=True,
            apply_to_output=True,
        ),
        # 4 Custom PII: Block execution if banned words are detected
        PIIMiddleware(
            pii_type="banned_words",
            detector=banned_words_pattern,
            strategy="block",  # Block the calls and # Raises an exception before calling LLM
            apply_to_input=True,
            apply_to_output=True
        ),
    ],
)

In [ ]:
print("--- Test: PII Redaction & Masking ---")

res = agent.invoke({"messages": [{"role": "user",
                                  "content": "My name is John Doe. My email is john.doe@example.com and card is 5105-1051-0510-5100. Please process my $50 order."
                                  }]
                    })
print("Agent Output:", res["messages"][-1].content)

In [ ]:
print("--- Test: Blocked Words ---")
try:
  res = agent.invoke({"messages": [{"role": "user",
                                    "content": "Please search for details regarding confidential status on secret_project."
                                    }]
                      })
except PIIDetectionError as err:
  print("Agent Output:", {err})